In [16]:
# ==============================================================================
# BLOCK 1: IMPORTS AND GRANDMASTER CORPUS DOWNLOAD (BOZON v0.7 RL)
# ==============================================================================
!pip install -q python-chess requests tqdm

import io
import math
import random
import time
import zipfile
import requests
import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.benchmark = True

print(f"[INFO] Initializing BOZON v0.7 (Self-Play & Reinforcement Learning)")
print(f"[INFO] Computing Device: {device}")

PGN_MENTOR_PLAYERS = ["Carlsen", "Kasparov", "Fischer", "Tal", "Karpov", "Nakamura"]

def fetch_pgn_mentor_corpus(players: list) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    combined_pgn = []
    for player in players:
        url = f"https://www.pgnmentor.com/players/{player}.zip"
        try:
            r = requests.get(url, headers=headers, timeout=30)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
                for fname in zf.namelist():
                    if fname.endswith(".pgn"):
                        text = zf.read(fname).decode("utf-8", errors="ignore")
                        combined_pgn.append(text)
                        print(f"[SUCCESS] Loaded {fname} ({len(text)//1024:,} KB)")
        except Exception as exc:
            print(f"[WARNING] Skipped {player}: {exc}")
    return "\n\n".join(combined_pgn)

pgn_data = fetch_pgn_mentor_corpus(PGN_MENTOR_PLAYERS)

[INFO] Initializing BOZON v0.7 (Self-Play & Reinforcement Learning)
[INFO] Computing Device: cuda
[SUCCESS] Loaded Carlsen.pgn (5,598 KB)
[SUCCESS] Loaded Kasparov.pgn (1,318 KB)
[SUCCESS] Loaded Fischer.pgn (537 KB)
[SUCCESS] Loaded Tal.pgn (1,421 KB)
[SUCCESS] Loaded Karpov.pgn (2,330 KB)
[SUCCESS] Loaded Nakamura.pgn (7,592 KB)


In [17]:
# ==============================================================================
# BLOCK 2: CANONICAL ENCODING & GPU-RESIDENT DATASET BUILDER
# ==============================================================================
CHANNEL_WEIGHTS = np.array([1.0, 3.1, 3.3, 5.0, 9.5, 0.0], dtype=np.float32)

def canonical_board_to_tensor(board: chess.Board) -> np.ndarray:
    """Encodes board into (18, 8, 8) uint8 tensor from ACTIVE player's view."""
    tensor = np.zeros((18, 8, 8), dtype=np.uint8)
    turn = board.turn
    flip = (turn == chess.BLACK)

    for piece_type in range(1, 7):
        for color in (chess.WHITE, chess.BLACK):
            mask = board.pieces_mask(piece_type, color)
            if not mask:
                continue
            is_mine = (color == turn)
            ch = (piece_type - 1) + (0 if is_mine else 6)
            while mask:
                lsb = mask & -mask
                sq = lsb.bit_length() - 1
                c_sq = (sq ^ 56) if flip else sq
                tensor[ch, 7 - (c_sq >> 3), c_sq & 7] = 1
                mask ^= lsb

    if board.is_check():
        tensor[12, :, :] = 1

    for move in board.legal_moves:
        to_sq = (move.to_square ^ 56) if flip else move.to_square
        tensor[13, 7 - (to_sq >> 3), to_sq & 7] = 1

    my_k = chess.BB_H8 if flip else chess.BB_H1
    my_q = chess.BB_A8 if flip else chess.BB_A1
    op_k = chess.BB_H1 if flip else chess.BB_H8
    op_q = chess.BB_A1 if flip else chess.BB_A8

    if bool(board.castling_rights & my_k):
        tensor[14, :, :] = 1
    if bool(board.castling_rights & my_q):
        tensor[15, :, :] = 1
    if bool(board.castling_rights & op_k):
        tensor[16, :, :] = 1
    if bool(board.castling_rights & op_q):
        tensor[17, :, :] = 1

    return tensor


def encode_move_canonical(move: chess.Move, turn: bool) -> int:
    if turn == chess.BLACK:
        return (move.from_square ^ 56) * 64 + (move.to_square ^ 56)
    return move.from_square * 64 + move.to_square


def compute_tensor_material_value(tensor_batch: np.ndarray) -> np.ndarray:
    counts = tensor_batch[:, :12, :, :].sum(axis=(2, 3)).astype(np.float32)
    my_mat = counts[:, 0:6] @ CHANNEL_WEIGHTS
    op_mat = counts[:, 6:12] @ CHANNEL_WEIGHTS
    return np.tanh((my_mat - op_mat) / 4.0).astype(np.float32)


def build_v07_datasets(pgn_text: str, gm_positions: int = 100000, aug_positions: int = 20000):
    t0 = time.time()
    train_gm_max = int(gm_positions * 0.90)
    val_max = gm_positions - train_gm_max
    total_train = train_gm_max + aug_positions

    tr_s = np.zeros((total_train, 18, 8, 8), dtype=np.uint8)
    tr_m = np.zeros((total_train,), dtype=np.int64)
    tr_out = np.zeros((total_train,), dtype=np.float32)
    tr_pmask = np.ones((total_train,), dtype=np.float32)

    va_s = np.zeros((val_max, 18, 8, 8), dtype=np.uint8)
    va_m = np.zeros((val_max,), dtype=np.int64)
    va_out = np.zeros((val_max,), dtype=np.float32)
    sample_fens = []

    pgn_io = io.StringIO(pgn_text)
    game_idx, tr_idx, va_idx = 0, 0, 0

    while tr_idx < train_gm_max or va_idx < val_max:
        game = chess.pgn.read_game(pgn_io)
        if game is None:
            break
        res = game.headers.get("Result", "*")
        if res not in ("1-0", "0-1", "1/2-1/2"):
            continue
        outcome_w = 1.0 if res == "1-0" else (-1.0 if res == "0-1" else 0.0)

        game_idx += 1
        is_val = (game_idx % 10 == 0)
        board = game.board()

        for ply, move in enumerate(game.mainline_moves()):
            state_t = canonical_board_to_tensor(board)
            move_i = encode_move_canonical(move, board.turn)
            out_active = outcome_w if board.turn == chess.WHITE else -outcome_w

            # Collect diverse middlegame/endgame FENs for Self-Play RL phase
            if 16 <= ply <= 65 and len(sample_fens) < 3000 and random.random() < 0.08:
                sample_fens.append(board.fen())

            if is_val and va_idx < val_max:
                va_s[va_idx], va_m[va_idx], va_out[va_idx] = state_t, move_i, out_active
                va_idx += 1
            elif not is_val and tr_idx < train_gm_max:
                tr_s[tr_idx], tr_m[tr_idx], tr_out[tr_idx] = state_t, move_i, out_active
                tr_idx += 1

            board.push(move)
            if tr_idx >= train_gm_max and va_idx >= val_max:
                break

    rng = np.random.default_rng(42)
    src_indices = rng.choice(tr_idx, size=aug_positions, replace=False)
    for k, src_i in enumerate(src_indices):
        dest_i = train_gm_max + k
        tr_s[dest_i] = tr_s[src_i].copy()
        tr_m[dest_i] = tr_m[src_i]
        tr_pmask[dest_i] = 0.0
        ch = int(rng.choice([1, 2, 3, 4, 7, 8, 9, 10]))
        tr_s[dest_i, ch, :, :] = 0

    tr_mat_val = compute_tensor_material_value(tr_s)
    va_mat_val = compute_tensor_material_value(va_s)

    tr_v = 0.80 * tr_mat_val + 0.20 * tr_out
    tr_v[train_gm_max:] = tr_mat_val[train_gm_max:]
    va_v = 0.80 * va_mat_val + 0.20 * va_out

    print(f"[SUCCESS] Prepared {total_train:,} train & {val_max:,} val positions in {time.time()-t0:.1f}s")
    print(f"[INFO] Collected {len(sample_fens):,} diverse positions for Self-Play RL rollouts.")
    return (
        torch.from_numpy(tr_s).to(device),
        torch.from_numpy(tr_m).to(device),
        torch.from_numpy(tr_v).unsqueeze(1).to(device),
        torch.from_numpy(tr_pmask).to(device),
        torch.from_numpy(va_s).to(device),
        torch.from_numpy(va_m).to(device),
        torch.from_numpy(va_v).unsqueeze(1).to(device),
        sample_fens
    )

tr_X, tr_y_pol, tr_y_val, tr_pmask, va_X, va_y_pol, va_y_val, rl_start_fens = build_v07_datasets(pgn_data)
del pgn_data

[SUCCESS] Prepared 110,000 train & 10,000 val positions in 14.4s
[INFO] Collected 3,000 diverse positions for Self-Play RL rollouts.


In [19]:
# ==============================================================================
# BLOCK 3: BOZON v0.7 ARCHITECTURE & STAGE 1 SUPERVISED WARM-START
# ==============================================================================
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.fc(x).unsqueeze(-1).unsqueeze(-1)


class SEResidualBlock(nn.Module):
    def __init__(self, channels: int, dropout_p: float = 0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.drop = nn.Dropout2d(p=dropout_p)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction=8)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.drop(out)
        out = self.se(self.bn2(self.conv2(out)))
        return F.relu(out + residual, inplace=True)


class BozonSEResNet(nn.Module):
    def __init__(self, num_blocks: int = 10, channels: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(18, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )
        self.res_tower = nn.Sequential(
            *[SEResidualBlock(channels, dropout_p=0.1) for _ in range(num_blocks)]
        )
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Dropout(p=0.25),
            nn.Linear(32 * 8 * 8, 4096)
        )
        self.value_conv = nn.Sequential(
            nn.Conv2d(channels, 8, kernel_size=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.Flatten()
        )
        self.value_mlp = nn.Sequential(
            nn.Linear(512 + 18, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.15),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
            nn.Tanh()
        )

    def forward(self, x: torch.Tensor):
        piece_counts = x[:, :12, :, :].sum(dim=(2, 3)) * 0.125
        meta_features = x[:, 12:, :, :].mean(dim=(2, 3))
        features = self.res_tower(self.stem(x))
        policy_logits = self.policy_head(features)
        v_spatial = self.value_conv(features)
        v_combined = torch.cat([v_spatial, piece_counts, meta_features], dim=1)
        return policy_logits, self.value_mlp(v_combined)


model = BozonSEResNet(num_blocks=10, channels=128).to(device)
policy_criterion = nn.CrossEntropyLoss(label_smoothing=0.05, reduction="none")
value_criterion = nn.MSELoss()

optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=3e-4)
epochs = 12
batch_size = 512
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

def evaluate_model_vram(model: nn.Module, X_val, y_pol_val, y_val_val, b_size: int = 512):
    model.eval()
    n_val = X_val.size(0)
    pol_loss_sum, val_mse_sum, top1, top3 = 0.0, 0.0, 0, 0
    ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()

    with torch.no_grad():
        for i in range(0, n_val, b_size):
            states = X_val[i:i+b_size].to(torch.float32)
            moves = y_pol_val[i:i+b_size]
            values = y_val_val[i:i+b_size]
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                p_logits, v_preds = model(states)
                pol_loss_sum += ce(p_logits, moves).item() * states.size(0)
                val_mse_sum += mse(v_preds, values).item() * states.size(0)
            _, top3_idx = torch.topk(p_logits, k=3, dim=1)
            top1 += (top3_idx[:, 0] == moves).sum().item()
            top3 += (top3_idx == moves.unsqueeze(1)).any(dim=1).sum().item()

    return pol_loss_sum / n_val, val_mse_sum / n_val, (top1 / n_val) * 100, (top3 / n_val) * 100


print("[STAGE 1] Running Supervised Pre-Training (12 Epochs)...")
n_train = tr_X.size(0)

for epoch in range(1, epochs + 1):
    t0 = time.time()
    model.train()
    perm = torch.randperm(n_train, device=device)

    for i in range(0, n_train, batch_size):
        idx = perm[i:i+batch_size]
        states = tr_X[idx].to(torch.float32)
        moves, values, pmask = tr_y_pol[idx], tr_y_val[idx], tr_pmask[idx]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            p_logits, v_preds = model(states)
            raw_p = policy_criterion(p_logits, moves)
            loss_p = (raw_p * pmask).sum() / (pmask.sum() + 1e-6)
            loss_v = value_criterion(v_preds, values)
            loss = loss_p + (4.0 * loss_v)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    scheduler.step()
    val_p, val_mse, val_top1, val_top3 = evaluate_model_vram(model, va_X, va_y_pol, va_y_val)
    if epoch in (1, 4, 8, 12):
        print(
            f"[PRETRAIN {epoch:02d}/{epochs:02d}] ({time.time()-t0:.1f}s) | "
            f"ValPol: {val_p:.3f} | ValMSE: {val_mse:.4f} | Top-1: {val_top1:.2f}% | Top-3: {val_top3:.2f}%"
        )

[STAGE 1] Running Supervised Pre-Training (12 Epochs)...
[PRETRAIN 01/12] (14.8s) | ValPol: 3.513 | ValMSE: 0.0630 | Top-1: 21.33% | Top-3: 39.62%
[PRETRAIN 04/12] (14.8s) | ValPol: 3.110 | ValMSE: 0.0304 | Top-1: 28.33% | Top-3: 47.95%
[PRETRAIN 08/12] (14.9s) | ValPol: 3.390 | ValMSE: 0.0266 | Top-1: 29.73% | Top-3: 48.60%
[PRETRAIN 12/12] (14.9s) | ValPol: 3.503 | ValMSE: 0.0261 | Top-1: 30.51% | Top-3: 49.22%


In [20]:
# ==============================================================================
# BLOCK 4: STAGE 2 - PARALLEL SELF-PLAY & TD-REINFORCEMENT LEARNING (v0.7 CORE)
# ==============================================================================
PIECE_VALUES = {
    chess.PAWN: 1.0, chess.KNIGHT: 3.1, chess.BISHOP: 3.3,
    chess.ROOK: 5.0, chess.QUEEN: 9.5, chess.KING: 0.0
}

def exact_tactical_utility(board: chess.Board, perspective: bool) -> float:
    """Computes exact material + checkmate + endgame king-cornering utility."""
    if board.is_checkmate():
        return -10.0 if board.turn == perspective else 10.0
    if board.is_stalemate() or board.is_insufficient_material():
        return 0.0

    my_m, op_m = 0.0, 0.0
    for pt, val in PIECE_VALUES.items():
        if val > 0:
            my_m += len(board.pieces(pt, perspective)) * val
            op_m += len(board.pieces(pt, not perspective)) * val
    diff = my_m - op_m

    # Endgame mate drive bonus
    if diff >= 3.0:
        ok = board.king(not perspective)
        if ok is not None:
            center_d = abs(chess.square_rank(ok) - 3.5) + abs(chess.square_file(ok) - 3.5)
            diff += center_d * 0.15
    return float(np.tanh(diff / 4.0))


def select_oracle_tactical_move(board: chess.Board, model_probs: np.ndarray) -> tuple:
    """
    Expert Iteration / RL Oracle:
    Evaluates top candidate moves + captures/checks using 2-ply lookahead with
    capture resolution to find the true optimal action & TD-target value.
    """
    turn = board.turn
    legal = list(board.legal_moves)

    # Pick top 6 policy moves + all checks and captures
    scored_priors = [(model_probs[encode_move_canonical(m, turn)], m) for m in legal]
    scored_priors.sort(key=lambda x: x[0], reverse=True)
    candidates = {m for _, m in scored_priors[:6]}
    for m in legal:
        if board.is_capture(m) or board.gives_check(m):
            candidates.add(m)

    best_move = scored_priors[0][1]
    best_td_val = -99.0

    for move in candidates:
        board.push(move)
        if board.is_checkmate():
            board.pop()
            return move, 1.0  # Mate-in-1 found!

        if board.is_stalemate() or board.is_insufficient_material():
            reply_val = 0.0
        else:
            # Opponent's best tactical response (with recapture check)
            worst_for_us = 99.0
            for opp_m in board.legal_moves:
                if board.is_capture(opp_m) or board.gives_check(opp_m):
                    board.push(opp_m)
                    if board.is_checkmate():
                        worst_for_us = -1.0
                        board.pop()
                        break
                    # Check if we can immediately recapture on ply 3!
                    best_recap = exact_tactical_utility(board, turn)
                    for recap_m in board.legal_moves:
                        if board.is_capture(recap_m):
                            board.push(recap_m)
                            u = exact_tactical_utility(board, turn)
                            if u > best_recap:
                                best_recap = u
                            board.pop()
                    if best_recap < worst_for_us:
                        worst_for_us = best_recap
                    board.pop()

            if worst_for_us == 99.0:
                worst_for_us = exact_tactical_utility(board, turn)
            reply_val = worst_for_us

        board.pop()
        # Small prior tie-breaker
        prior_bonus = 0.08 * model_probs[encode_move_canonical(move, turn)]
        if (reply_val + prior_bonus) > best_td_val:
            best_td_val = reply_val + prior_bonus
            best_move = move

    return best_move, float(np.clip(best_td_val, -1.0, 1.0))


def run_batched_self_play_rl(
    model: nn.Module,
    start_fens: list,
    rl_iterations: int = 6,
    parallel_games: int = 64,
    steps_per_iter: int = 10
):
    """
    Runs Batched Self-Play Rollouts on GPU and updates the network via
    Expert Iteration (Policy Cross-Entropy on tactical oracle moves + TD Value MSE).
    """
    rl_optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    ce_loss_fn = nn.CrossEntropyLoss()
    mse_loss_fn = nn.MSELoss()

    print(f"[STAGE 2 - RL] Starting {rl_iterations} Self-Play RL Iterations ({parallel_games} parallel boards)...")

    for it in range(1, rl_iterations + 1):
        t0 = time.time()
        model.eval()

        # Initialize 64 parallel boards from diverse middlegame/endgame FENs
        boards = [chess.Board(random.choice(start_fens)) for _ in range(parallel_games)]
        replay_states, replay_best_moves, replay_td_values = [], [], []
        checkmates_found = 0

        for _ in range(steps_per_iter):
            active_boards = [b for b in boards if not b.is_game_over()]
            if not active_boards:
                break

            batch_tensors = np.stack([canonical_board_to_tensor(b) for b in active_boards])
            t_gpu = torch.tensor(batch_tensors, dtype=torch.float32, device=device)

            with torch.no_grad():
                logits, _ = model(t_gpu)
                probs_batch = torch.softmax(logits, dim=1).cpu().numpy()

            for b_idx, b in enumerate(active_boards):
                best_m, td_val = select_oracle_tactical_move(b, probs_batch[b_idx])
                if td_val >= 0.99:
                    checkmates_found += 1

                replay_states.append(batch_tensors[b_idx])
                replay_best_moves.append(encode_move_canonical(best_m, b.turn))
                replay_td_values.append(td_val)

                # Advance self-play board (80% best move, 20% exploration to create tactical traps)
                if random.random() < 0.80:
                    b.push(best_m)
                else:
                    b.push(random.choice(list(b.legal_moves)))

        # RL Gradient Update on collected Self-Play Replay Buffer + GM Replay Anchor
        model.train()
        rl_S = torch.tensor(np.array(replay_states), dtype=torch.float32, device=device)
        rl_M = torch.tensor(np.array(replay_best_moves), dtype=torch.long, device=device)
        rl_V = torch.tensor(np.array(replay_td_values), dtype=torch.float32, device=device).unsqueeze(1)

        # Mix with a batch of GM positions to prevent catastrophic forgetting of openings
        gm_idx = torch.randint(0, tr_X.size(0) - 20000, (rl_S.size(0),), device=device)
        comb_S = torch.cat([rl_S, tr_X[gm_idx].to(torch.float32)], dim=0)
        comb_M = torch.cat([rl_M, tr_y_pol[gm_idx]], dim=0)
        comb_V = torch.cat([rl_V, tr_y_val[gm_idx]], dim=0)

        perm = torch.randperm(comb_S.size(0), device=device)
        rl_p_loss, rl_v_loss, steps = 0.0, 0.0, 0

        for i in range(0, comb_S.size(0), 256):
            p_idx = perm[i:i+256]
            rl_optimizer.zero_grad(set_to_none=True)
            p_out, v_out = model(comb_S[p_idx])
            lp = ce_loss_fn(p_out, comb_M[p_idx])
            lv = mse_loss_fn(v_out, comb_V[p_idx])
            (lp + 3.0 * lv).backward()
            rl_optimizer.step()
            rl_p_loss += lp.item()
            rl_v_loss += lv.item()
            steps += 1

        print(
            f"[RL ITER {it:02d}/{rl_iterations:02d}] ({time.time()-t0:.1f}s) | "
            f"Rollout States: {len(replay_states)} | Tactical Mates Found: {checkmates_found} | "
            f"RL Policy Loss: {rl_p_loss/steps:.3f} | TD-Value MSE: {rl_v_loss/steps:.4f}"
        )

run_batched_self_play_rl(model, rl_start_fens, rl_iterations=6, parallel_games=64, steps_per_iter=10)

weights_path = "bozon_v0.7_rl.pt"
torch.save(model.state_dict(), weights_path)
checksum = float(next(model.parameters()).sum().item())
print(f"\n[SUCCESS] Saved BOZON v0.7 RL Weights to '{weights_path}' | Checksum: {checksum:.6f}")

[STAGE 2 - RL] Starting 6 Self-Play RL Iterations (64 parallel boards)...
[RL ITER 01/06] (8.2s) | Rollout States: 640 | Tactical Mates Found: 66 | RL Policy Loss: 1.580 | TD-Value MSE: 0.1730
[RL ITER 02/06] (7.0s) | Rollout States: 630 | Tactical Mates Found: 95 | RL Policy Loss: 1.562 | TD-Value MSE: 0.1597
[RL ITER 03/06] (5.8s) | Rollout States: 637 | Tactical Mates Found: 100 | RL Policy Loss: 1.592 | TD-Value MSE: 0.1486
[RL ITER 04/06] (8.2s) | Rollout States: 635 | Tactical Mates Found: 83 | RL Policy Loss: 1.608 | TD-Value MSE: 0.1474
[RL ITER 05/06] (6.2s) | Rollout States: 638 | Tactical Mates Found: 72 | RL Policy Loss: 1.560 | TD-Value MSE: 0.1434
[RL ITER 06/06] (7.5s) | Rollout States: 634 | Tactical Mates Found: 50 | RL Policy Loss: 1.669 | TD-Value MSE: 0.1373

[SUCCESS] Saved BOZON v0.7 RL Weights to 'bozon_v0.7_rl.pt' | Checksum: -40.224838


In [24]:
# ==============================================================================
# BLOCK 5: DECISIVE CHECKMATE TEST (VERIFY FAST WIN AGAINST BOT)
# ==============================================================================
model.eval()

def play_fast_checkmate_test():
    board = chess.Board()
    print("[ARENA] Testing BOZON v0.7 RL (White) vs ELO-150 Bot (Black)...")

    for move_num in range(1, 60):
        if board.is_game_over():
            break

        # White: BOZON v0.7 RL
        t = torch.tensor(canonical_board_to_tensor(board), dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, _ = model(t)
            probs = torch.softmax(logits.squeeze(0), dim=0).cpu().numpy()

        best_move, expected_val = select_oracle_tactical_move(board, probs)
        san = board.san(best_move)
        board.push(best_move)

        if board.is_game_over():
            print(f"Move {move_num:02d} | BOZON plays {san:<6} -> CHECKMATE! Result: {board.result()}")
            break

        # Black: Weak Bot
        bot_move = random.choice(list(board.legal_moves))
        board.push(bot_move)

        if move_num % 5 == 0:
            print(f"Move {move_num:02d} | BOZON: {san:<6} | Expected Eval: {expected_val:+.2f}")

    outcome = board.outcome()
    term_reason = outcome.termination if outcome else "Unknown"
    print(f"[FINAL RESULT] {board.result()} in {board.fullmove_number} moves | Status: {term_reason}")

play_fast_checkmate_test()

[ARENA] Testing BOZON v0.7 RL (White) vs ELO-150 Bot (Black)...
Move 05 | BOZON: Bg2    | Expected Eval: +0.48
Move 10 | BOZON: Qb3    | Expected Eval: +1.00
Move 15 | BOZON: Re1    | Expected Eval: +1.00
Move 20 | BOZON: Ne4    | Expected Eval: +1.00
Move 25 | BOZON: Nc5+   | Expected Eval: +1.00
Move 27 | BOZON plays Rc4#   -> CHECKMATE! Result: 1-0
[FINAL RESULT] 1-0 in 27 moves | Status: Termination.CHECKMATE
